In [4]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

# pretrained model
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.eval()

# preprocessing (no fixed resize!)
preprocess = weights.transforms()

def run(h, w):
    x = torch.rand(1,3,h,w)  # random image
    with torch.no_grad():
        y = model(x)
    print(f"Input: {h}x{w} -> Output shape:", y.shape)

run(224,224)
run(320,240)
run(512,512)

Input: 224x224 -> Output shape: torch.Size([1, 1000])
Input: 320x240 -> Output shape: torch.Size([1, 1000])
Input: 512x512 -> Output shape: torch.Size([1, 1000])


In [5]:
import torch
from torchvision.models import vgg16, VGG16_Weights

model = vgg16(weights=VGG16_Weights.DEFAULT).eval()

x = torch.rand(1,3,224,224)

with torch.no_grad():
    y = model(x)

print(y.shape)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\smi09/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100%|██████████| 528M/528M [00:22<00:00, 24.6MB/s] 


torch.Size([1, 1000])


In [10]:
x = torch.rand(1,3,257,257)

with torch.no_grad():
    y = model(x)

print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FixedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,16,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)

        # assume input = 32x32
        # after pool -> 16x16
        # after pool -> 8x8
        self.fc = nn.Linear(32*8*8, 10)

    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = self.fc(x)
        return x

model = FixedCNN()

In [13]:
x = torch.rand(1,3,32,32)
y = model(x)
print(y.shape)

torch.Size([1, 10])


In [14]:
x = torch.rand(1,3,40,40)
y = model(x)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x3200 and 2048x10)